In [ ]:
pip install scikit-learn

In [ ]:
import os
from pathlib import Path
import torch
from PIL import Image
import numpy as np
import torchvision.transforms as transforms
import pandas as pd
from tqdm import tqdm

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

import tarfile
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

In [ ]:
model_path = "./models/densenet121_pneumonia_full.pth" # change to path of model we want to use

# Validation data (HAS labels)
val_folder = Path("./val_data/val_224")
val_label_csv = "./val_data/classification_val.csv"
val_output_csv = "val_predictions.csv"

# Test data (NO labels)
test_folder = Path("./test_data/test_224")
test_output_csv = "test_predictions.csv"

In [ ]:
model = torch.load(model_path, map_location="cpu", weights_only=False)
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_preprocessed_tensor(path):
    img = Image.open(path).convert("RGB")
    return eval_transform(img)


Loading model...


FileNotFoundError: [Errno 2] No such file or directory: './models/densenet121_pneumonia_full.pth'

In [ ]:
# Test on one image
tensor_train = load_preprocessed_tensor("./val_data/val_224/09f0ac0a-2933-45fc-a937-c0da67f5fd0d.png").unsqueeze(0)
output = model(tensor_train)
prob = torch.sigmoid(output).item()
print(prob)

In [ ]:
# Evaluate with metrics (validation set)
def evaluate_with_labels(folder, label_csv, output_csv):
    print(f"\nEvaluating Validation Set: {folder}")

    df_labels = pd.read_csv(label_csv)
    label_dict = {str(pid): int(label) for pid, label in zip(df_labels.patientId, df_labels.label)}

    image_files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])

    all_ids = []
    all_preds = []
    all_true = []
    all_probs = []

    with torch.no_grad():
        for filename in tqdm(image_files):
            patient_id = filename.replace(".png", "")
            img_path = folder / filename

            tensor = load_preprocessed_tensor(img_path).unsqueeze(0)  # shape [1,3,224,224]
            output = model(tensor)                                   # shape [1,1]
            prob = torch.sigmoid(output).item()                      # float 0-1
            pred = int(prob >= 0.5)                                  # 0 or 1

            all_ids.append(patient_id)
            all_preds.append(pred)
            all_true.append(label_dict[patient_id])
            all_probs.append(prob)

    # Save 0/1 predictions
    df_out = pd.DataFrame({
        "patientId": all_ids,
        "true_label": all_true,
        "prediction": all_preds,
        "probability": all_probs
    })
    df_out.to_csv(output_csv, index=False)
    print(f"Saved validation predictions to {output_csv}")

    # Metrics
    y_true = all_true
    y_pred = all_preds

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print("\nValidation Metrics")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm)

    return df_out

In [ ]:
# Predict on Test set
def evaluate_no_labels(folder, output_csv):
    print(f"\nEvaluating Test Set: {folder}")

    image_files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])

    all_ids = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for filename in tqdm(image_files):
            patient_id = filename.replace(".png", "")
            img_path = folder / filename

            tensor = load_preprocessed_tensor(img_path).unsqueeze(0)
            output = model(tensor)
            prob = torch.sigmoid(output).item()
            pred = int(prob >= 0.5)

            all_ids.append(patient_id)
            all_preds.append(pred)
            all_probs.append(prob)

    df_out = pd.DataFrame({
        "patientId": all_ids,
        "prediction": all_preds,
        "probability": all_probs
    })
    df_out.to_csv(output_csv, index=False)
    print(f"Saved test predictions to {output_csv}")

    return df_out

In [ ]:
evaluate_with_labels(val_folder, val_label_csv, val_output_csv)
evaluate_no_labels(test_folder, test_output_csv)

# Evaluate ResNet

In [ ]:
model_path = "./models/resnet50_pneumonia_full.pth" # change to path of model we want to use
model = torch.load(model_path, map_location="cpu", weights_only=False)
model.eval()

evaluate_with_labels(val_folder, val_label_csv, val_output_csv)
evaluate_no_labels(test_folder, test_output_csv)

# Evaluate Faster-RCNN


In [ ]:
pip install pycocotools

6060.99s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [ ]:
model_path = "./models/fasterrcnn_resume.pth"

test_folder = Path("./test_data/test_512")           # folder with test images
val_folder = Path("./val_data/val_512")             # folder with validation images
val_labels_csv = "./val_data/bboxes_val.csv"
output_val_csv = "val_fasterrcnn_preds.csv"
output_test_csv = "test_fasterrcnn_preds.csv"

conf_threshold = 0.5
default_h, default_w = 512, 512

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

mps


In [ ]:
num_classes = 2
model = fasterrcnn_resnet50_fpn(weights=None, num_classes=num_classes)
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

def load_image(path):
    img = Image.open(path).convert("RGB")
    return F.to_tensor(img)

In [ ]:
def predict_folder(image_folder, output_csv):
    image_folder = Path(image_folder)
    all_data = []
    image_files = sorted([f for f in image_folder.glob("*.png")])
    print(f"Running predictions on {len(image_files)} images in {image_folder}")

    with torch.no_grad():
        for img_path in tqdm(image_files):
            patient_id = img_path.stem
            img_tensor = load_image(img_path).to(device).unsqueeze(0)

            outputs = model(img_tensor)
            output = outputs[0]

            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()

            for box, score in zip(boxes, scores):
                if score < conf_threshold:
                    continue
                x_min, y_min, x_max, y_max = box
                all_data.append({
                    "patientId": patient_id,
                    "x_min": x_min,
                    "y_min": y_min,
                    "x_max": x_max,
                    "y_max": y_max,
                    "score": score
                })

    df_out = pd.DataFrame(all_data)
    df_out.to_csv(output_csv, index=False)
    print(f"Saved predictions to {output_csv}")
    return df_out, image_files

In [ ]:
def evaluate_coco(pred_df, image_files, gt_csv=None, output_coco_json="coco_preds.json"):
    image_id_map = {f.stem: i+1 for i, f in enumerate(image_files)}

    coco_preds = []
    for _, row in pred_df.iterrows():
        patient_id = row["patientId"]
        image_id = image_id_map.get(patient_id)
        if image_id is None:
            continue
        x_min = row["x_min"]
        y_min = row["y_min"]
        width = row["x_max"] - row["x_min"]
        height = row["y_max"] - row["y_min"]
        score = row["score"]
        coco_preds.append({
            "image_id": image_id,
            "category_id": 1,
            "bbox": [x_min, y_min, width, height],
            "score": score
        })

    with open(output_coco_json, "w") as f:
        json.dump(coco_preds, f)
    print(f"Saved COCO predictions JSON to {output_coco_json}")

    if gt_csv:
        df_gt = pd.read_csv(gt_csv)
        images = []
        annotations = []
        ann_id = 1
        for patient_id, group in df_gt.groupby("patientId"):
            img_id = image_id_map.get(patient_id)
            if img_id is None:
                continue
            images.append({
                "id": img_id,
                "file_name": f"{patient_id}.png",
                "height": default_h,
                "width": default_w
            })
            for _, row in group.iterrows():
                x_min = row["x_min"]
                y_min = row["y_min"]
                width = row["x_max"] - row["x_min"]
                height = row["y_max"] - row["y_min"]
                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": 1,
                    "bbox": [x_min, y_min, width, height],
                    "area": width * height,
                    "iscrowd": 0
                })
                ann_id += 1
        coco_gt_json = "coco_gt.json"
        coco_gt_dict = {
            "images": images,
            "annotations": annotations,
            "categories": [{"id": 1, "name": "pneumonia"}]
        }
        with open(coco_gt_json, "w") as f:
            json.dump(coco_gt_dict, f)
        print(f"Saved COCO GT JSON to {coco_gt_json}")

        coco_gt_obj = COCO(coco_gt_json)
        coco_dt_obj = coco_gt_obj.loadRes(output_coco_json)
        coco_eval = COCOeval(coco_gt_obj, coco_dt_obj, iouType='bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

In [ ]:
val_df, val_images = predict_folder(val_folder, output_test_csv)
evaluate_coco(val_df, val_images, gt_csv=val_labels_csv, output_coco_json="val_coco_preds.json")


Running predictions on 6405 images in val_data/val_512


  0%|          | 0/6405 [02:51<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
test_df, test_images = predict_folder(test_folder, output_val_csv)
evaluate_coco(test_df, test_images, gt_csv=None, output_coco_json="test_coco_preds.json")

# YOLO Predictions


In [ ]:
pip install ultralytics


7352.78s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached ultralytics-8.3.237-py3-none-any.whl.metadata (37 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached pyyaml-6.0.3-cp39-cp39-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached polars-1.36.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.36.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (1.5 kB)
  Using cached charset_normalizer-3.4.4-cp39-cp39-macosx_10_9_universal2.whl.metadata (37 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
Using cached ultralytics-8.3.237-py3-none-any.whl (1.2 MB)
Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl (37.9 MB)
Using cached polars-1.36.1-py3-none-any.whl (802

In [ ]:
tar_path = "./models/runs.tar.gz"
extract_dir = Path("yolo_runs")
extract_dir.mkdir(exist_ok=True)

with tarfile.open(tar_path) as tar:
    tar.extractall(path=extract_dir)

print("Extracted YOLO runs to:", extract_dir)

Extracted YOLO runs to: yolo_runs


In [ ]:
weights_path = extract_dir /"runs/detect/train/weights/best.pt"
assert weights_path.exists(), f"Could not find weights at {weights_path}"

In [ ]:
val_img_dir = Path("./data/images/val_512")
val_csv = Path("./val_data/bboxes_val.csv")

test_img_dir = Path("./test_data/test_512")

val_labels_dir = Path("val_labels")
val_labels_dir.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(val_csv)

for img_file in val_img_dir.iterdir():
    if img_file.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
        continue
    pid = img_file.stem
    bboxes = df[df["patientId"] == pid]

    label_lines = []
    for _, row in bboxes.iterrows():
        x_min, y_min, x_max, y_max = row[["x_min", "y_min", "x_max", "y_max"]]
        w = x_max - x_min
        h = y_max - y_min
        x_c = x_min + w / 2
        y_c = y_min + h / 2

        img_w, img_h = 512, 512
        x_c /= img_w
        y_c /= img_h
        w /= img_w
        h /= img_h

        label_lines.append(f"0 {x_c} {y_c} {w} {h}")

    label_file = val_labels_dir / f"{pid}.txt"
    label_file.write_text("\n".join(label_lines))

print(f"Converted CSV to YOLO txt labels in {val_labels_dir}")

Converted CSV to YOLO txt labels in val_labels


In [ ]:
val_yaml = f"""
path: ./data          # root folder
names: ["pneumonia"]
nc: 1
train: dummy_train     # just a placeholder
val: images/val_512    # relative to path
"""

val_yaml_path = Path("val_data.yaml")
val_yaml_path.write_text(val_yaml)

model = YOLO(str(weights_path))


Conf = 0.1 (runs/detect/val)
- mAP50 = 0.368, mAP50-95 = 0.173

Conf = 0.25 (runs/detect/val3)
- mAP50 = 0.375, mAP50-95 = 0.191

Conf = 0.15 (runs/detect/val4)
- mAP50 = 0.37, mAP50-95 = 0.19

Conf = 0.2 (runs/detect/val5)
- mAP50 = 0.375, mAP50-95 = 0.186

In [ ]:
val_results = model.val(data=str(val_yaml_path), batch=16, device='mps', conf=0.2)

Ultralytics 8.3.237 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M3)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 181.1±109.1 MB/s, size: 101.1 KB)
val: Scanning /Users/amandalu/School/CornellTech/AML/Final/data/labels/val_512.cache... 6405 images, 4987 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6405/6405 5.8Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 401/401 3.7it/s 1:50<0.3ss
                   all       6405       2274      0.482      0.345      0.375      0.186
Speed: 0.2ms preprocess, 3.8ms inference, 0.0ms loss, 4.7ms postprocess per image
Results saved to /Users/amandalu/School/CornellTech/AML/Final/runs/detect/val5


In [ ]:
precision, recall, map50, map5095 = val_results.mean_results()
print("Precision:", precision)
print("Recall:", recall)
print("mAP50:", map50)
print("mAP50-95:", map5095)

Precision: 0.4076984594792528
Recall: 0.4309586631486368
mAP50: 0.36880316841528205
mAP50-95: 0.1588210929793552


In [ ]:
output_csv = "yolo_val_predictions.csv"

with open(output_csv, "w") as f:
    f.write("patientId,PredictionString\n")

    results_stream = model.predict(
        source=str(val_img_dir),
        imgsz=512,
        conf=0.1,
        device='mps',
        stream=True
    )

    for r in results_stream:
        image_id = Path(r.path).stem
        pred_strings = []

        if len(r.boxes) > 0:
            for box, score in zip(r.boxes.xyxy, r.boxes.conf):
                x_min, y_min, x_max, y_max = box.cpu().numpy()


                scale_factor = 1024 / 512
                x_min *= scale_factor
                y_min *= scale_factor
                x_max *= scale_factor
                y_max *= scale_factor
                width = x_max - x_min
                height = y_max - y_min

                pred_strings.append(f"{score:.4f} {int(x_min)} {int(y_min)} {int(width)} {int(height)}")

        f.write(f"{image_id},{ ' '.join(pred_strings) }\n")

print(f"Saved memory-efficient submission to {output_csv}")


image 1/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/0010f549-b242-4e94-87a8-57d79de215fc.png: 512x512 (no detections), 86.0ms
image 2/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/001916b8-3d30-4935-a5d1-8eaddb1646cd.png: 512x512 (no detections), 23.1ms
image 3/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/0087bd3a-55a7-4045-b111-b018fa52d361.png: 512x512 3 pneumonias, 14.2ms
image 4/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/008b69b2-446a-43dd-9ba2-9ccff8f3da41.png: 512x512 (no detections), 13.7ms
image 5/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/008c19e8-a820-403a-930a-bc74a4053664.png: 512x512 (no detections), 14.4ms
image 6/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/0092d9c5-26b6-4e66-b196-49b2224ab8d1.png: 512x512 (no detections), 13.4ms
image 7/6405 /Users/amandalu/School/CornellTech/AML/Final/data/images/val_512/00c0b293-48e

In [ ]:
yolo_val_csv = "./val_data/bboxes_val.csv"
df = pd.read_csv(yolo_val_csv)

df["width"]  = df["x_max"] - df["x_min"]
df["height"] = df["y_max"] - df["y_min"]


yolo_df = df[["patientId", "x_min", "y_min", "width", "height"]]

scale_factor = 1024 / 512
yolo_df[["x_min", "y_min", "width", "height"]] *= scale_factor
yolo_df["confidence"] = 1.0

all_patient_ids = [p.stem for p in val_img_dir.glob("*.png")]

pred_groups = yolo_df.groupby("patientId")

submission_rows = []
for patient_id in all_patient_ids:
    if patient_id in pred_groups.groups:
        group = pred_groups.get_group(patient_id)
        pred_strings = []
        for _, row in group.iterrows():
            pred_strings.append(f"{row['confidence']} {row['x_min']} {row['y_min']} {row['width']} {row['height']}")
        submission_rows.append([patient_id, " ".join(pred_strings)])
    else:
        submission_rows.append([patient_id, ""])

submission_df = pd.DataFrame(submission_rows, columns=["patientId", "PredictionString"])
submission_df.sort_values("patientId", inplace=True)
submission_csv = "val_true_values.csv"
submission_df.to_csv(submission_csv, index=False)

print(f"Validation submission file saved to {submission_csv}")

/var/folders/8b/5y5qj_5s0_9gn3bqqfgt0jq80000gn/T/ipykernel_73009/3758763619.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yolo_df[["x_min", "y_min", "width", "height"]] *= scale_factor
/var/folders/8b/5y5qj_5s0_9gn3bqqfgt0jq80000gn/T/ipykernel_73009/3758763619.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yolo_df["confidence"] = 1.0


Validation submission file saved to val_true_values.csv


Make final submission file

In [ ]:
output_csv = "yolo_submission.csv"

with open(output_csv, "w") as f:
    f.write("patientId,PredictionString\n")

    results_stream = model.predict(
        source=str(test_img_dir),
        imgsz=512,
        conf=0.15,
        device='mps',
        stream=True
    )

    for r in results_stream:
        image_id = Path(r.path).stem
        pred_strings = []

        if len(r.boxes) > 0:
            for box, score in zip(r.boxes.xyxy, r.boxes.conf):
                x_min, y_min, x_max, y_max = box.cpu().numpy()

                scale_factor = 1024 / 512
                x_min *= scale_factor
                y_min *= scale_factor
                x_max *= scale_factor
                y_max *= scale_factor
                width = x_max - x_min
                height = y_max - y_min

                pred_strings.append(f"{score:.4f} {int(x_min)} {int(y_min)} {int(width)} {int(height)}")

        f.write(f"{image_id},{ ' '.join(pred_strings) }\n")

print(f"Saved memory-efficient submission to {output_csv}")



image 1/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/0000a175-0e68-4ca4-b1af-167204a7e0bc.png: 512x512 (no detections), 135.0ms
image 2/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/0005d3cc-3c3f-40b9-93c3-46231c3eb813.png: 512x512 (no detections), 16.4ms
image 3/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/000686d7-f4fc-448d-97a0-44fa9c5d3aa6.png: 512x512 (no detections), 14.5ms
image 4/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/000e3a7d-c0ca-4349-bb26-5af2d8993c3d.png: 512x512 (no detections), 8.4ms
image 5/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/00100a24-854d-423d-a092-edcf6179e061.png: 512x512 (no detections), 19.7ms
image 6/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/0015597f-2d69-4bc7-b642-5b5e01534676.png: 512x512 (no detections), 12.0ms
image 7/3000 /Users/amandalu/School/CornellTech/AML/Final/test_data/test_512/001b0c51-c7b3-45

In [ ]:
import gc
gc.collect()

386182